In [ ]:
import json
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

import holoviews as hv
from bokeh.models import HoverTool

hv.extension('bokeh')
renderer = hv.renderer('bokeh')

In [ ]:
# GraphML file generated by Gephi
INPUT_GRAPHML = 'bipartite_podcasts_layout_threshold=1.graphml'

# File to save visualization to (without `.html` extension)
OUTPUT_HTML = 'holoviews'

# Scaling factors for visualization glyphs
EDGE_SCALING = 0.5
NODE_SCALING = 1

# Label font size range (pt) — proportional to node size
LABEL_FONT_SIZE_MIN = 2
LABEL_FONT_SIZE_MAX = 15

# Bounding box of the network visualization
GRAPH_EXTENTS = (-800, -800, 800, 800)

CATEGORY_COLORS = {
  "Intellectual": "#00acc6",
  "Celebrity": "#018700",
  "Podcast": "#8c3bff",
  "Athlete": "#e6a500",
  "Politician": "#ff7ed1",
  "Other": "#6b004f",
  "Influencer": "#573b00",
  "Comedian": "#005659",
  "Political Pundit": "#15e18c"
}

In [ ]:
def format_views(n):
    """Format view count as compact string, e.g. 78.4M, 1.2K."""
    if pd.isna(n) or n is None:
        return "—"
    n = int(float(n))
    if n >= 1_000_000:
        return f"{n / 1_000_000:.0f}M"
    if n >= 1_000:
        return f"{n / 1_000:.0f}K"
    return str(n)

def hide_hook(plot, element):
    """Hide frame around plot: https://discourse.holoviz.org/t/remove-frame-around-plot/4023/7"""
    plot.handles["xaxis"].visible = False
    plot.handles["yaxis"].visible = False
    plot.handles["plot"].border_fill_color = None
    plot.handles["plot"].background_fill_color = None
    plot.handles["plot"].outline_line_color = None

## 1. Load laid out GraphML file and convert to Pandas DataFrames for nodes and edges

In [ ]:
# Load the laid-out GraphML file saved from Gephi into a NetworkX graph
G = nx.read_graphml(path = INPUT_GRAPHML)

# Extract node and edge information from NetworkX graph into raw DataFrames
_nodes_df = pd.DataFrame.from_dict(
    dict(G.nodes(data=True)), orient='index'
).reset_index(drop = True)
_edges_df = nx.to_pandas_edgelist(G = G)

# Set column order of nodes DataFrame and ignore irrelevant attributes
nodes_df = _nodes_df[['x', 'y', 'size', 'views', 'views_root_3', 'label', 'category']]
# Create dict that maps node name to a numeric index
node_names = sorted(nodes_df['label'])
name_to_index = dict( zip(node_names, np.arange(len(node_names))))
# Create a column in the nodes DataFrame for the numeric index (this is necessary for HoloViews visualization)
nodes_df.insert(loc = 2, column = 'index', value = nodes_df['label'].map(name_to_index))

# Remove the unnecessary "id" column from the edges DataFrame
edges_df = _edges_df.drop('id', axis = 'columns')
# Instead of keeping track of the channel names of the source and target nodes, keep track of the numeric index of each source and target node
edges_df['source'] = edges_df['source'].map(name_to_index)
edges_df['target'] = edges_df['target'].map(name_to_index)
# For visualization purposes, scale the edge weight by a certain value
edges_df['weight'] /= EDGE_SCALING

# Make label font size proportional to node size
size_min, size_max = nodes_df['size'].min(), nodes_df['size'].max()
size_range = size_max - size_min
nodes_df['text_font_size'] = (
    LABEL_FONT_SIZE_MIN
    + (nodes_df['size'] - size_min) / size_range * (LABEL_FONT_SIZE_MAX - LABEL_FONT_SIZE_MIN)
)

# Make view count have nice format
nodes_df["views_formatted"] = nodes_df["views"].apply(format_views)

# Set node color based on category
nodes_df["category_color"] = nodes_df["category"].map(CATEGORY_COLORS)

In [ ]:
nodes_df

In [ ]:
edges_df

## 2. Convert to HoloViews DataFrame and save visualization

In [ ]:
# Convert node DataFrame to HoloViews object
hv_nodes = hv.Nodes(nodes_df)

# Create HoloViews Graph object from nodes and edges, with x and y limits
# bounded by `GRAPH_EXTENTS`
hv_graph = hv.Graph((edges_df, hv_nodes), extents=GRAPH_EXTENTS)

# Specify Holoviews options
hv_graph.opts(
    node_radius="size",)

# Generate visualization
renderer.save(hv_graph, OUTPUT_HTML + '_default')

In [ ]:
# Convert node DataFrame to HoloViews object
hv_nodes = hv.Nodes(nodes_df)

# Create HoloViews Graph object from nodes and edges, with x and y limits
# bounded by `GRAPH_EXTENTS`
hv_graph = hv.Graph((edges_df, hv_nodes), extents=GRAPH_EXTENTS)

labels = hv.Labels(nodes_df, ['x', 'y'], ['label', 'text_font_size'])
hv_graph_with_labels = (hv_graph * labels.opts(text_font_size='text_font_size', text_color='black', bgcolor='gray'))

# Define custom hover tooltip (views_formatted is e.g. 78M, 1.2K)
hover = HoverTool(tooltips=[("Name", "@label"), ("Total Views", "@views_formatted"), ("Category", "@category")])

# Specify Holoviews options
hv_graph.opts(
    node_radius="size",
    node_color="category_color",
    node_hover_fill_color="#DF0000",
    node_hover_fill_alpha=1,
    node_line_color="#ffffff",
    node_hover_line_color="#DF0000",
    edge_color="#bbbbbb",
    edge_line_width="weight",
    edge_hover_line_color="#DF0000",
    edge_hover_alpha=1,
    edge_selection_alpha=1,
    edge_selection_line_color="#DF0000",
    responsive=True,
    aspect="equal",
    active_tools=["pan", "wheel_zoom"],
    tools=[hover],
    toolbar=None,
    directed=True,
    arrowhead_length=0.002,
)

hv_graph_with_labels.opts(
    bgcolor="white",
    xticks=0,
    yticks=0,
    xlabel="",
    ylabel="",
    hooks=[hide_hook],
)

# Generate visualization
renderer.save(hv_graph_with_labels, OUTPUT_HTML)

# Center the plot in the viewport and scale correctly
out_path = Path(OUTPUT_HTML).resolve().parent / (Path(OUTPUT_HTML).name + ".html")
if out_path.exists():
    html = out_path.read_text(encoding="utf-8")
    center_style = (
        "<style>"
        "html, body { margin: 0; min-height: 100vh; "
        "display: flex; justify-content: center; align-items: center; }"
        "</style>"
    )
    if "</head>" in html and center_style not in html:
        html = html.replace("</head>", center_style + "\n</head>", 1)
        out_path.write_text(html, encoding="utf-8")